# Dynamic model selection

**Module:** Module 3 — Production-Oriented Agent Design  
**Notebook:** `notebooks/module-3/3.4_dynamic_models.ipynb`

## Purpose
Use model-call middleware to choose a model at runtime based on the request or conversation state.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- Long conversations need deliberate context management, not simply unlimited message history.
- Human-in-the-loop checkpoints are a control boundary for actions with external side effects.
- Dynamic model/prompt/tool selection can reduce cost and tailor capabilities, but adds complexity.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



### Implementation step 1

Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

from dotenv import load_dotenv

load_dotenv()

### Implementation step 2

Imports the libraries and APIs needed by the following example: from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse; from langchain.chat_models import init_chat_model; from typing import Callable. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse; from langchain.chat_models import init_chat_model; from typing import Callable. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

### Implementation step 3

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example.

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

### Implementation step 4

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

### Implementation step 5

Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response["messages"][-1].response_metadata["model_name"])

### Implementation step 6

Imports the libraries and APIs needed by the following example: from langchain.messages import AIMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import AIMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

### Implementation step 7

Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response["messages"][-1].response_metadata["model_name"])